<a href="https://colab.research.google.com/github/Adyan213/Hands-On-ML/blob/main/Chapter_13_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import tensorflow as tf

root='https://ai.stanford.edu/~amaas/data/sentiment/'
filename='aclImdb_v1.tar.gz'
filepath=tf.keras.utils.get_file(filename, root+filename, extract=True,
                                 cache_dir='.')
path=Path(filepath)/'aclImdb'

In [2]:
def tree(path, level=0, indent=4, max_files=3):
  if level==0:
    print(f'{path}/')
    level+=1
  sub_paths=sorted(path.iterdir())
  sub_dirs=[sub_path for sub_path in sub_paths if sub_path.is_dir()]
  filepaths=[sub_path for sub_path in sub_paths if not sub_path in sub_dirs]
  indent_str=" "*4*level

  for sub_dir in sub_dirs:
    print(f'{indent_str}{sub_dir.name}/')
    tree(sub_dir, level+1, indent)
  for filepath in filepaths[:max_files]:
    print(f'{indent_str}{filepath.name}')
  if len(filepaths)>max_files:
    print(f'{indent_str}...')

In [3]:
tree(path)

datasets/aclImdb_v1_extracted/aclImdb/
    test/
        neg/
            0_2.txt
            10000_4.txt
            10001_1.txt
            ...
        pos/
            0_10.txt
            10000_7.txt
            10001_9.txt
            ...
        labeledBow.feat
        urls_neg.txt
        urls_pos.txt
    train/
        neg/
            0_3.txt
            10000_4.txt
            10001_4.txt
            ...
        pos/
            0_9.txt
            10000_8.txt
            10001_10.txt
            ...
        unsup/
            0_0.txt
            10000_0.txt
            10001_0.txt
            ...
        labeledBow.feat
        unsupBow.feat
        urls_neg.txt
        ...
    README
    imdb.vocab
    imdbEr.txt


In [4]:
from numpy import test
def review_paths(dirpath):
  return [str(path) for path in dirpath.glob('*.txt')]

train_pos=review_paths(path/'train'/'pos')
train_neg=review_paths(path/'train'/'neg')
test_valid_pos=review_paths(path/'test'/'pos')
test_valid_neg=review_paths(path/'test'/'neg')

len(train_pos), len(train_neg), len(test_valid_pos), len(test_valid_neg)

(12500, 12500, 12500, 12500)

In [5]:
import numpy as np
np.random.shuffle(test_valid_pos)

test_pos=test_valid_pos[:5000]
valid_pos=test_valid_pos[5000:]
test_neg=test_valid_neg[:5000]
valid_neg=test_valid_neg[5000:]

In [6]:
def imdb_dataset(filepaths_positive, filepaths_negative):
  reviews=[]
  labels=[]
  for filepaths, label in [(filepaths_positive, 1), (filepaths_negative, 0)]:
    for filepath in filepaths:
      with open(filepath) as review_file:
        reviews.append(review_file.read())
        labels.append(label)
  return tf.data.Dataset.from_tensor_slices((
      tf.constant(reviews), tf.constant(labels)
  ))

In [7]:
for X, y in imdb_dataset(train_pos, train_neg).take(3):
  print(X)
  print(y)
  print()

tf.Tensor(b"Fritz Lang's German medieval saga continues in Die Nibelungen: Kriemhilds Rache (1924).Kriemhild (Margarete Schoen) wants to avenge her murdered husband, Siegfried.Her brother is too weak to bring the murdered, Hagen, to justice.Kriemhild marries Etzel, the King of the Huns (Rudolf Klein-Rogge).She gives birth to a child, and invites her brothers for a party.Etzel and the other Huns should kil Hagen, but he is protected by his brothers.We see a huge battle of life and death begins, and she sets the whole place on fire.Eventually Hagen is dead, and so is Kriemhild.These movies deal with great themes, such as revenge and undying love.Sure we have later seen some better movies made of those topics, but this was one of the early ones.", shape=(), dtype=string)
tf.Tensor(1, shape=(), dtype=int32)

tf.Tensor(b'Surely one the French films of the decade so far, a taut, atmospheric thriller making full use of the lead characters hearing impediment to use sound in a way rarely explor

In [8]:
%timeit -r1 for X, y in imdb_dataset(train_pos, train_neg).repeat(10): pass

1min 5s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [9]:
def imdb_dataset(filepaths_positive, filepaths_negative, n_read_threads=5):
  dataset_neg=tf.data.TextLineDataset(filepaths_negative, num_parallel_reads=n_read_threads)
  dataset_neg=dataset_neg.map(lambda review: (review, 0))
  dataset_pos=tf.data.TextLineDataset(filepaths_positive, num_parallel_reads=n_read_threads)
  dataset_pos=dataset_pos.map(lambda review: (review, 1))
  return tf.data.Dataset.concatenate(dataset_pos, dataset_neg)

In [10]:
%timeit -r1 for X, y in imdb_dataset(train_pos, train_neg).repeat(10): pass

1min 36s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [11]:
%timeit -r1 for X, y in imdb_dataset(train_pos, train_neg).cache().repeat(10): pass

1min 11s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [12]:
batch_size=32

train_set=imdb_dataset(train_pos, train_neg).shuffle(25000).batch(batch_size).prefetch(1)
valid_set=imdb_dataset(valid_pos, valid_neg).batch(batch_size).prefetch(1)
test_set=imdb_dataset(test_pos, test_neg).batch(batch_size).prefetch(1)

In [13]:
max_tokens=1000
sample_reviews=train_set.map(lambda review, label: review)
text_vectorization=tf.keras.layers.TextVectorization(max_tokens=max_tokens,
                                                     output_mode='tf_idf')
text_vectorization.adapt(sample_reviews)

In [14]:
text_vectorization.get_vocabulary()[:10]

['[UNK]',
 np.str_('the'),
 np.str_('and'),
 np.str_('a'),
 np.str_('of'),
 np.str_('to'),
 np.str_('is'),
 np.str_('in'),
 np.str_('it'),
 np.str_('i')]

In [15]:
tf.random.set_seed(42)
model=tf.keras.Sequential([
    text_vectorization,
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(loss='binary_crossentropy', optimizer='nadam', metrics=['accuracy'])
model.fit(train_set, validation_data=valid_set, epochs=5)

Epoch 1/5
    781/Unknown 9s 6ms/step - accuracy: 0.7893 - loss: 0.4820

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.8236 - loss: 0.4341 - val_accuracy: 0.8441 - val_loss: 0.3770
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.8525 - loss: 0.3683 - val_accuracy: 0.8500 - val_loss: 0.3632
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.8691 - loss: 0.3238 - val_accuracy: 0.8429 - val_loss: 0.3812
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.8883 - loss: 0.2733 - val_accuracy: 0.8470 - val_loss: 0.3756
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9158 - loss: 0.2115 - val_accuracy: 0.8425 - val_loss: 0.3875


In [16]:
def compute_mean_embedding(inputs):
    not_pad = tf.math.count_nonzero(inputs, axis=-1)
    n_words = tf.math.count_nonzero(not_pad, axis=-1, keepdims=True)
    sqrt_n_words = tf.math.sqrt(tf.cast(n_words, tf.float32))
    return tf.reduce_sum(inputs, axis=1) / sqrt_n_words

another_example = tf.constant([[[1., 2., 3.], [4., 5., 0.], [0., 0., 0.]],
                               [[6., 0., 0.], [0., 0., 0.], [0., 0., 0.]]])
compute_mean_embedding(another_example)

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[3.535534 , 4.9497476, 2.1213205],
       [6.       , 0.       , 0.       ]], dtype=float32)>

In [17]:
embedding_size=20
tf.random.set_seed(42)

text_vectorization=tf.keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='int'
)
text_vectorization.adapt(sample_reviews)

model=tf.keras.Sequential([
    text_vectorization,
    tf.keras.layers.Embedding(input_dim=max_tokens, output_dim=embedding_size,
                              mask_zero=True),
    tf.keras.layers.Lambda(compute_mean_embedding),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [18]:
model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
model.fit(train_set, epochs=5, validation_data=valid_set)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'lambda' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


782/782 ━━━━━━━━━━━━━━━━━━━━ 21s 21ms/step - accuracy: 0.6333 - loss: 0.6253 - val_accuracy: 0.5502 - val_loss: 1.0687
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - accuracy: 0.7565 - loss: 0.5055 - val_accuracy: 0.5945 - val_loss: 0.8830
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - accuracy: 0.7924 - loss: 0.4480 - val_accuracy: 0.8413 - val_loss: 0.3950
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 18s 21ms/step - accuracy: 0.8108 - loss: 0.4186 - val_accuracy: 0.7234 - val_loss: 0.5164
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 20ms/step - accuracy: 0.8183 - loss: 0.4105 - val_accuracy: 0.8530 - val_loss: 0.3585


In [19]:
import tensorflow_datasets as tfds

# To resolve potential dependency issues that prevent tfds.load from being available,
# it's often helpful to reinstall tensorflow and tensorflow-datasets to get compatible versions.
# Uncomment and run the following in a separate cell or once here:
!pip install --upgrade tensorflow tensorflow-datasets

datasets=tfds.load(name='imdb_reviews')
train_set, test_set=datasets['train'], datasets['test']

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.X4K26V_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.X4K26V_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.X4K26V_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


In [20]:
for example in train_set.take(1):
  print(example['text'])
  print(example['label'])

tf.Tensor(b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it.", shape=(), dtype=string)
tf.Tensor(0, shape=(), dtype=int64)
